# Importando bibliotecas


In [ ]:
import pandas as pd
import plotly.express as px

# Importando dados

In [ ]:
df = pd.read_excel('../data/raw/Relatório de Produção - Completo - Dados.xlsx')

# Visualizando dados

In [ ]:
df

# Tratando dados

## Base Diária

In [ ]:
df_diario = df.copy()

# Tratando o cabeçalho
df_diario.columns = df_diario.iloc[1]
df_diario = df_diario.iloc[2:].reset_index(drop=True)
df_diario.columns.name = None

# Convertendo as métricas para valores numéricos
colunas_float = [
    "Eficiencia Plan",
    "Hora Hora Wht (6to6)",
    "Veloc Stand",
    "Veloc Real",
    "Qtd Teorica Real",
    "Corte de Gota %",
    "Objetivo %",
    "Qtd Objetivo",
    "Empacotado %",
    "Qtd Empacotado",
    "Qtd Rejeicao",
    "Rejeição %",
]

for coluna in colunas_float:
    df_diario[coluna] = pd.to_numeric(
        df_diario[coluna]
        .astype("string")
        .str.strip()
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )

# Agrupando os registros horários pela produção identificada pela OP
desempenho_empacotamento_diario = (
    df_diario
    .groupby(
        "OP Vertech",
        as_index=False,
    )
    .agg({
        # Informações descritivas da produção
        "Data Wht (dia)": "first",
        "Maquina": "first",
        "Prefixo": "first",

        # Métricas da produção
        "Objetivo %": "min",
        "Empacotado %": "mean",
        "Rejeição %": "mean",
    })
)

# Calculando o desempenho depois da rejeição
desempenho_empacotamento_diario[
    "Emp - Rejeitado %"
] = (
    desempenho_empacotamento_diario["Empacotado %"]
    * (
        1
        - desempenho_empacotamento_diario["Rejeição %"]
    )
)

display(desempenho_empacotamento_diario)

## Base Hora a Hora


In [ ]:
# Tratando cabeçalho
df.columns = df.iloc[1]
df = df.iloc[2:].reset_index(drop=True)
df.columns.name = None

# Criando as variáveis de data e hora do dia produtivo (07h até 06h)
df["Hora Hora Wht (6to6)"] = pd.to_numeric(
    df["Hora Hora Wht (6to6)"],
    errors="coerce"
)

df["Data Wht (dia)"] = pd.to_datetime(
    df["Data Wht (dia)"],
    dayfirst=True,
    errors="coerce"
).dt.normalize()

df["Data_metrica"] = (
    df["Data Wht (dia)"]
    + pd.to_timedelta(
        df["Hora Hora Wht (6to6)"],
        unit="h"
    )
)

# Ordem operacional: 07h = 0, 08h = 1, ..., 23h = 16, 00h = 17, ..., 06h = 23
df["Ordem_Hora_Producao"] = (
    (df["Hora Hora Wht (6to6)"] - 7) % 24
)

# Chave técnica para ordenar vários dias produtivos sem alterar Data Wht (dia)
df["Data_Hora_Ordem_Producao"] = (
    df["Data Wht (dia)"]
    + pd.to_timedelta(
        df["Ordem_Hora_Producao"],
        unit="h"
    )
)

df["Hora_Producao"] = (
    df["Hora Hora Wht (6to6)"]
    .astype("Int64")
    .astype("string")
    .str.zfill(2)
    + ":00"
)

ORDEM_HORAS_PRODUCAO = [
    f"{hora:02d}:00"
    for hora in list(range(7, 24)) + list(range(0, 7))
]

# transformando variaveis numericas corretamente
colunas_numericas = [
    'Veloc Stand',
    'Veloc Real',
    'Qtd Teorica Real',
    'Corte de Gota %',
    'Objetivo %',
    'Qtd Objetivo',
    'Empacotado %',
    'Qtd Empacotado',
    'Rejeição %',
    'Qtd Rejeicao'
]

for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(df[coluna], errors='coerce')
    
# criando variavel "Emp - Rej"
df["Emp - Rej %"] = (
    df["Empacotado %"] - df["Rejeição %"]
)

df["Qtd Emp - Qtd Rej"] = (
    df["Qtd Empacotado"] - df["Qtd Rejeicao"]
)

# criando variavel de gap entre "Emp - Rej %" e "Objetivo %"
df['Gap_Obj %'] = (df['Emp - Rej %'] - df['Objetivo %'])
df['Qtd Gap_Obj'] = (df['Qtd Emp - Qtd Rej'] - df['Qtd Objetivo'])

display(df)

# Análises

## Anotações 



Insights:
- Variável "Qtd Teorica Real" tem comportamento similar ao "Veloc Real" por OP hora a hora
    - Motivo: "Qtd Teorica Real" é a quantidade que teoricamente poderia ser produzida durante uma hora considerando a velocidade real informada
    - "Qtd Teorica Real" = "Veloc Real" × 60

- Empacotado < Rejeitado
    - Notado que há OPs com qtd rejeitada maior que qtd empacotada, não faz sentido
    - Ação: verificar se alguma vírgula ou algo do tipo na planilha, que pode estar afetando na leitura em pandas

Futuras análises: 
- Calcular um valor que represente a performance de uma OP
    - Mostrar as OPs com menores performances
    - Ver variáveis Empacotado, Objetivo e Rejeitado de hora em hora
        - Fazer um contador de quantas vezes OP ficou abaixo do objetivo?
        - Fazer alguma coisa com rejeitado também!

## Informações das variáveis

In [ ]:
df.info()

## Cobertura de tempo

In [ ]:
cobertura_tempo = df.sort_values(
    ["Data_Hora_Ordem_Producao"]
)

display(
    cobertura_tempo.iloc[0][
        ["Data Wht (dia)", "Hora_Producao"]
    ].rename("Inicio do período produtivo")
)

display(
    cobertura_tempo.iloc[-1][
        ["Data Wht (dia)", "Hora_Producao"]
    ].rename("Fim do período produtivo")
)

## Correlação de variáveis

In [ ]:
# vairaveis numericas
df_numerico = df.select_dtypes(include="number")

# removendo variaveis categoricas ou não-variáveis
df_numerico = df_numerico.drop(columns={
    'Corte de Gota %'
})

# calculo matriz de correlacao
matriz_correlacao = df_numerico.corr(method="pearson")

# plotagem da matriz
fig = px.imshow(
    matriz_correlacao,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Matriz de Correlação das Variáveis"
)

fig.update_layout(
    xaxis_title="Variáveis",
    yaxis_title="Variáveis",
    width=1000,
    height=800
)

fig.show()

## Validando chave "OP Vertech"

**Pergunta a responder:** Cada OP em "OP Vertech" se refere a uma máquina e um prefixo? Ou seja, uma produção?

In [ ]:
validacao_chave_op = (
    df
    .groupby("OP Vertech", as_index=False)
    .agg(
        Quantidade_de_Maquinas=("Maquina", "nunique"),
        Quantidade_de_Prefixos=("Prefixo", "nunique"),
    )
)

ops_inconsistentes = validacao_chave_op.query(
    "Quantidade_de_Maquinas > 1 "
    "or Quantidade_de_Prefixos > 1"
)

print(
    "OPs associadas a mais de uma máquina ou prefixo:",
    len(ops_inconsistentes),
)

display(ops_inconsistentes)

## Validando valores da variável "Qtd Rejeicao" por OP

In [ ]:
# Seleciona as variáveis necessárias sem alterar o df original
base_rejeicao = df[
    [
        "OP Vertech",
        "Maquina",
        "Prefixo",
        "Data Wht (dia)",
        "Data_metrica",
        "Data_Hora_Ordem_Producao",
        "Hora Hora Wht (6to6)",
        "Hora_Producao",
        "Qtd Rejeicao",
    ]
].copy()

# Garante os tipos corretos
base_rejeicao["Data_metrica"] = pd.to_datetime(
    base_rejeicao["Data_metrica"],
    errors="coerce",
)

base_rejeicao["Qtd Rejeicao"] = pd.to_numeric(
    base_rejeicao["Qtd Rejeicao"],
    errors="coerce",
)

# Remove registros que não podem ser associados a uma OP e hora
base_rejeicao = base_rejeicao.dropna(
    subset=["OP Vertech", "Data_metrica"]
)

# Ordena cada OP seguindo o ciclo produtivo 07h até 06h
base_rejeicao = base_rejeicao.sort_values(
    ["OP Vertech", "Data_Hora_Ordem_Producao"]
)

In [ ]:
duplicidades_op_hora = base_rejeicao[
    base_rejeicao.duplicated(
        subset=[
            "OP Vertech",
            "Data Wht (dia)",
            "Hora Hora Wht (6to6)",
        ],
        keep=False,
    )
]

print(
    "Registros duplicados para a mesma OP e hora:",
    len(duplicidades_op_hora),
)

display(duplicidades_op_hora)

In [ ]:
# Total de horas registradas para cada OP
total_horas_por_op = (
    base_rejeicao
    .groupby(
        "OP Vertech",
        as_index=False,
    )
    .agg(
        Quantidade_Total_de_Horas=(
            "Data_Hora_Ordem_Producao",
            "nunique",
        )
    )
)

# Uma linha para cada valor de rejeição encontrado em cada OP
resumo_valores_rejeicao = (
    base_rejeicao
    .groupby(
        ["OP Vertech", "Qtd Rejeicao"],
        dropna=False,
        as_index=False,
    )
    .agg(
        Maquina=("Maquina", "first"),
        Prefixo=("Prefixo", "first"),

        Frequencia_de_Horas=(
            "Data_Hora_Ordem_Producao",
            "nunique",
        ),

        Lista_de_Horas=(
            "Hora_Producao",
            lambda horas: horas.drop_duplicates().tolist(),
        ),

        Lista_de_Datas_e_Horas=(
            "Data_metrica",
            lambda datas: (
                datas
                .drop_duplicates()
                .dt.strftime("%d/%m/%Y %H:%M")
                .tolist()
            ),
        ),
    )
    .merge(
        total_horas_por_op,
        how="left",
        on="OP Vertech",
    )
)

# Percentual das horas da OP em que cada valor apareceu
resumo_valores_rejeicao[
    "Percentual_das_Horas"
] = (
    resumo_valores_rejeicao["Frequencia_de_Horas"]
    / resumo_valores_rejeicao["Quantidade_Total_de_Horas"]
    * 100
)

# Organiza a ordem das colunas e das linhas
resumo_valores_rejeicao = (
    resumo_valores_rejeicao[
        [
            "OP Vertech",
            "Maquina",
            "Prefixo",
            "Qtd Rejeicao",
            "Frequencia_de_Horas",
            "Quantidade_Total_de_Horas",
            "Percentual_das_Horas",
            "Lista_de_Horas",
            "Lista_de_Datas_e_Horas",
        ]
    ]
    .sort_values(
        ["OP Vertech", "Qtd Rejeicao"]
    )
    .reset_index(drop=True)
)

display(resumo_valores_rejeicao)

In [ ]:
ops_unicas = df["OP Vertech"].dropna().unique()

for op in ops_unicas:
    dados_op = df[df["OP Vertech"] == op]

    maquina = dados_op["Maquina"].iloc[0]
    prefixo = dados_op["Prefixo"].iloc[0]

    contagem_rejeicao = (
        dados_op["Qtd Rejeicao"]
        .value_counts(dropna=False)
        .rename_axis("Qtd Rejeicao")
        .reset_index(name="Quantidade de Horas")
        .sort_values("Qtd Rejeicao")
    )

    print(
        f"OP: {op} | "
        f"Máquina: {maquina} | "
        f"Prefixo: {prefixo}"
    )

    display(contagem_rejeicao)

## Variação de "Emp - Rej %" por OP hora a hora

In [ ]:
'''df_analise = df.copy()
df_analise.sort_values(by='Data_Hora_Ordem_Producao', inplace=True)

fig = px.line(
    df_analise,
    x='Hora_Producao',
    y='Emp - Rej %',
    title='Variação de "Emp - Rej %" por OP hora a hora',
    color='OP Vertech',
    markers=True,
    category_orders={"Hora_Producao": ORDEM_HORAS_PRODUCAO},
    labels={
        'Hora_Producao': 'Hora do dia produtivo'
    },
    hover_data=[
        'Emp - Rej %',
        'Qtd Empacotado',
        'Rejeição %',
        'Qtd Rejeicao',
        'Objetivo %',
        'Qtd Objetivo'
    ]
)

fig.show()'''


df_analise = df.copy()

df_analise.sort_values(
    by="Data_Hora_Ordem_Producao",
    inplace=True
)

fig = px.line(
    df_analise,
    x="Hora_Producao",
    y="Emp - Rej %",
    title='Variação de "Emp - Rej %" por OP hora a hora',
    color="OP Vertech",
    markers=True,
    category_orders={
        "Hora_Producao": ORDEM_HORAS_PRODUCAO
    },
    labels={
        "Hora_Producao": "Hora do dia produtivo"
    },
    hover_data=[
        "Emp - Rej %",
        "Qtd Empacotado",
        "Rejeição %",
        "Qtd Rejeicao",
        "Objetivo %",
        "Qtd Objetivo",
    ],
)

# Faixas visuais dos turnos, seguindo a ordem do dia produtivo
faixas_turnos = [
    {
        "turno": "Turno A",
        "inicio": -0.5,
        "fim": 7,
        "hora_rotulo": "10:00",
        "cor": "#D6EAF8",
    },
    {
        "turno": "Turno B",
        "inicio": 7,
        "fim": 15,
        "hora_rotulo": "18:00",
        "cor": "#AED6F1",
    },
    {
        "turno": "Turno C",
        "inicio": 15,
        "fim": 23.5,
        "hora_rotulo": "02:00",
        "cor": "#85C1E9",
    },
]

# Adiciona as faixas coloridas e os nomes dos turnos
for faixa in faixas_turnos:
    fig.add_vrect(
        x0=faixa["inicio"],
        x1=faixa["fim"],
        fillcolor=faixa["cor"],
        opacity=0.35,
        line_width=0,
        layer="below",
    )

    fig.add_annotation(
        x=faixa["hora_rotulo"],
        y=0.98,
        xref="x",
        yref="paper",
        text=faixa["turno"],
        showarrow=False,
        font={
            "size": 12,
            "color": "#34495E",
        },
        bgcolor="rgba(255, 255, 255, 0.70)",
        borderpad=3,
    )

# Linhas pontilhadas nas mudanças de turno:
# Turno B às 14h e Turno C às 22h
limites_turnos = [7, 15]

for limite in limites_turnos:
    fig.add_shape(
        type="line",
        x0=limite,
        x1=limite,
        y0=0,
        y1=1,
        xref="x",
        yref="paper",
        line={
            "color": "rgba(52, 73, 94, 0.75)",
            "width": 1.5,
            "dash": "dot",
        },
        layer="above",
    )

fig.show()

In [ ]:
# Ordena uma única vez, sem modificar o df original
df_analise = df.sort_values(
    "Data_Hora_Ordem_Producao"
)

# Turno, início, fim, posição do rótulo e cor
faixas_turnos = [
    ("Turno A", -0.5, 7, "10:00", "#D6EAF8"),
    ("Turno B", 7, 15, "18:00", "#AED6F1"),
    ("Turno C", 15, 23.5, "02:00", "#85C1E9"),
]

# Cria um gráfico para cada OP
for op_vertech, dados_op in df_analise.groupby("OP Vertech"):
    fig = px.line(
        dados_op,
        x="Hora_Producao",
        y="Emp - Rej %",
        markers=True,
        color_discrete_sequence=["#3366CC"],
        category_orders={
            "Hora_Producao": ORDEM_HORAS_PRODUCAO
        },
        title=(
            f'Variação de "Emp - Rej %" da '
            f"OP {op_vertech} hora a hora"
        ),
        labels={
            "Hora_Producao": "Hora do dia produtivo"
        },
        hover_data=[
            "OP Vertech",
            "Qtd Empacotado",
            "Rejeição %",
            "Qtd Rejeicao",
            "Objetivo %",
            "Qtd Objetivo",
        ],
    )

    # Adiciona as faixas e os rótulos dos turnos
    for turno, inicio, fim, hora_rotulo, cor in faixas_turnos:
        fig.add_vrect(
            x0=inicio,
            x1=fim,
            fillcolor=cor,
            opacity=0.35,
            line_width=0,
            layer="below",
        )

        fig.add_annotation(
            x=hora_rotulo,
            y=0.98,
            xref="x",
            yref="paper",
            text=turno,
            showarrow=False,
            font={
                "size": 12,
                "color": "#34495E",
            },
            bgcolor="rgba(255, 255, 255, 0.70)",
            borderpad=3,
        )

    # Linhas pontilhadas às 14h e às 22h
    for limite in (7, 15):
        fig.add_shape(
            type="line",
            x0=limite,
            x1=limite,
            y0=0,
            y1=1,
            xref="x",
            yref="paper",
            line={
                "color": "rgba(52, 73, 94, 0.75)",
                "width": 1.5,
                "dash": "dot",
            },
            layer="above",
        )

    fig.update_yaxes(
        tickformat=".0%"
    )

    fig.update_layout(
        title_x=0.5,
        height=600,
        width=1000,
        showlegend=False,
        margin={
            "l": 80,
            "r": 60,
            "t": 90,
            "b": 70,
        },
    )

    fig.show()

## Variação da quantidade empacotada por OP hora a hora

In [ ]:
'''df_analise = df.copy()
df_analise.sort_values(by='Data_Hora_Ordem_Producao', inplace=True)

fig = px.line(
    df_analise,
    x='Hora_Producao',
    y='Empacotado %',
    title='Variação de "Empacotado" por OP hora a hora',
    color='OP Vertech',
    markers=True,
    category_orders={"Hora_Producao": ORDEM_HORAS_PRODUCAO},
    labels={
        'Hora_Producao': 'Hora do dia produtivo'
    },
    hover_data=[
        'Qtd Empacotado',
        'Rejeição %',
        'Qtd Rejeicao',
        'Objetivo %',
        'Qtd Objetivo'
    ]
)

fig.show()'''


df_analise = df.copy()

df_analise.sort_values(
    by="Data_Hora_Ordem_Producao",
    inplace=True
)

fig = px.line(
    df_analise,
    x="Hora_Producao",
    y="Empacotado %",
    title='Variação de "Empacotado" por OP hora a hora',
    color="OP Vertech",
    markers=True,
    category_orders={
        "Hora_Producao": ORDEM_HORAS_PRODUCAO
    },
    labels={
        "Hora_Producao": "Hora do dia produtivo"
    },
    hover_data=[
        "Qtd Empacotado",
        "Rejeição %",
        "Qtd Rejeicao",
        "Objetivo %",
        "Qtd Objetivo",
    ],
)

# Faixas visuais dos turnos, seguindo a ordem do dia produtivo
faixas_turnos = [
    {
        "turno": "Turno A",
        "inicio": -0.5,
        "fim": 7,
        "hora_rotulo": "10:00",
        "cor": "#D6EAF8",
    },
    {
        "turno": "Turno B",
        "inicio": 7,
        "fim": 15,
        "hora_rotulo": "18:00",
        "cor": "#AED6F1",
    },
    {
        "turno": "Turno C",
        "inicio": 15,
        "fim": 23.5,
        "hora_rotulo": "02:00",
        "cor": "#85C1E9",
    },
]

# Adiciona as faixas coloridas e os nomes dos turnos
for faixa in faixas_turnos:
    fig.add_vrect(
        x0=faixa["inicio"],
        x1=faixa["fim"],
        fillcolor=faixa["cor"],
        opacity=0.35,
        line_width=0,
        layer="below",
    )

    fig.add_annotation(
        x=faixa["hora_rotulo"],
        y=0.98,
        xref="x",
        yref="paper",
        text=faixa["turno"],
        showarrow=False,
        font={
            "size": 12,
            "color": "#34495E",
        },
        bgcolor="rgba(255, 255, 255, 0.70)",
        borderpad=3,
    )

# Linhas pontilhadas somente às 14h e às 22h
limites_turnos = [7, 15]

for limite in limites_turnos:
    fig.add_shape(
        type="line",
        x0=limite,
        x1=limite,
        y0=0,
        y1=1,
        xref="x",
        yref="paper",
        line={
            "color": "rgba(52, 73, 94, 0.75)",
            "width": 1.5,
            "dash": "dot",
        },
        layer="above",
    )

fig.show()

In [ ]:
# Ordena uma única vez, sem modificar o df original
df_analise = df.sort_values(
    "Data_Hora_Ordem_Producao"
)

# Turno, início, fim, posição do rótulo e cor
faixas_turnos = [
    ("Turno A", -0.5, 7, "10:00", "#D6EAF8"),
    ("Turno B", 7, 15, "18:00", "#AED6F1"),
    ("Turno C", 15, 23.5, "02:00", "#85C1E9"),
]

# Cria um gráfico separado para cada OP
for op_vertech, dados_op in df_analise.groupby("OP Vertech"):
    fig = px.line(
        dados_op,
        x="Hora_Producao",
        y="Empacotado %",
        markers=True,
        color_discrete_sequence=["#3366CC"],
        category_orders={
            "Hora_Producao": ORDEM_HORAS_PRODUCAO
        },
        title=(
            f'Variação de "Empacotado %" da '
            f"OP {op_vertech} hora a hora"
        ),
        labels={
            "Hora_Producao": "Hora do dia produtivo"
        },
        hover_data=[
            "OP Vertech",
            "Qtd Empacotado",
            "Rejeição %",
            "Qtd Rejeicao",
            "Objetivo %",
            "Qtd Objetivo",
        ],
    )

    # Adiciona as faixas e os rótulos dos turnos
    for turno, inicio, fim, hora_rotulo, cor in faixas_turnos:
        fig.add_vrect(
            x0=inicio,
            x1=fim,
            fillcolor=cor,
            opacity=0.35,
            line_width=0,
            layer="below",
        )

        fig.add_annotation(
            x=hora_rotulo,
            y=0.98,
            xref="x",
            yref="paper",
            text=turno,
            showarrow=False,
            font={
                "size": 12,
                "color": "#34495E",
            },
            bgcolor="rgba(255, 255, 255, 0.70)",
            borderpad=3,
        )

    # Linhas pontilhadas às 14h e às 22h
    for limite in (7, 15):
        fig.add_shape(
            type="line",
            x0=limite,
            x1=limite,
            y0=0,
            y1=1,
            xref="x",
            yref="paper",
            line={
                "color": "rgba(52, 73, 94, 0.75)",
                "width": 1.5,
                "dash": "dot",
            },
            layer="above",
        )

    # Formata os valores do eixo Y como percentuais
    fig.update_yaxes(
        tickformat=".0%"
    )

    fig.update_layout(
        title_x=0.5,
        height=600,
        width=1000,
        showlegend=False,
        margin={
            "l": 80,
            "r": 60,
            "t": 90,
            "b": 70,
        },
    )

    fig.show()

## Variação de "Rejeição" por OP hora a hora


In [ ]:
df_analise = df.copy()

df_analise.sort_values(
    by="Data_Hora_Ordem_Producao",
    inplace=True
)

fig = px.line(
    df_analise,
    x="Hora_Producao",
    y="Rejeição %",
    title='Variação de "Rejeição" por OP hora a hora',
    color="OP Vertech",
    markers=True,
    category_orders={
        "Hora_Producao": ORDEM_HORAS_PRODUCAO
    },
    labels={
        "Hora_Producao": "Hora do dia produtivo"
    },
    hover_data=[
        "Qtd Rejeicao",
        "Empacotado %",
        "Qtd Empacotado",
        "Objetivo %",
        "Qtd Objetivo",
    ],
)

# Faixas visuais dos turnos, seguindo a ordem do dia produtivo
faixas_turnos = [
    {
        "turno": "Turno A",
        "inicio": -0.5,
        "fim": 7,
        "hora_rotulo": "10:00",
        "cor": "#D6EAF8",
    },
    {
        "turno": "Turno B",
        "inicio": 7,
        "fim": 15,
        "hora_rotulo": "18:00",
        "cor": "#AED6F1",
    },
    {
        "turno": "Turno C",
        "inicio": 15,
        "fim": 23.5,
        "hora_rotulo": "02:00",
        "cor": "#85C1E9",
    },
]

# Adiciona as faixas coloridas e os nomes dos turnos
for faixa in faixas_turnos:
    fig.add_vrect(
        x0=faixa["inicio"],
        x1=faixa["fim"],
        fillcolor=faixa["cor"],
        opacity=0.35,
        line_width=0,
        layer="below",
    )

    fig.add_annotation(
        x=faixa["hora_rotulo"],
        y=0.98,
        xref="x",
        yref="paper",
        text=faixa["turno"],
        showarrow=False,
        font={
            "size": 12,
            "color": "#34495E",
        },
        bgcolor="rgba(255, 255, 255, 0.70)",
        borderpad=3,
    )

# Linhas pontilhadas nas mudanças de turno:
# Turno B às 14h e Turno C às 22h
limites_turnos = [7, 15]

for limite in limites_turnos:
    fig.add_shape(
        type="line",
        x0=limite,
        x1=limite,
        y0=0,
        y1=1,
        xref="x",
        yref="paper",
        line={
            "color": "rgba(52, 73, 94, 0.75)",
            "width": 1.5,
            "dash": "dot",
        },
        layer="above",
    )

fig.show()

In [ ]:
# Ordena uma única vez, sem modificar o df original
df_analise = df.sort_values(
    "Data_Hora_Ordem_Producao"
)

# Turno, início, fim, posição do rótulo e cor
faixas_turnos = [
    ("Turno A", -0.5, 7, "10:00", "#D6EAF8"),
    ("Turno B", 7, 15, "18:00", "#AED6F1"),
    ("Turno C", 15, 23.5, "02:00", "#85C1E9"),
]

# Cria um gráfico separado para cada OP
for op_vertech, dados_op in df_analise.groupby("OP Vertech"):
    fig = px.line(
        dados_op,
        x="Hora_Producao",
        y="Rejeição %",
        markers=True,
        color_discrete_sequence=["#3366CC"],
        category_orders={
            "Hora_Producao": ORDEM_HORAS_PRODUCAO
        },
        title=(
            f'Variação de "Rejeição %" da '
            f"OP {op_vertech} hora a hora"
        ),
        labels={
            "Hora_Producao": "Hora do dia produtivo"
        },
        hover_data=[
            "OP Vertech",
            "Qtd Rejeicao",
            "Empacotado %",
            "Qtd Empacotado",
            "Objetivo %",
            "Qtd Objetivo",
        ],
    )

    # Adiciona as faixas e os rótulos dos turnos
    for turno, inicio, fim, hora_rotulo, cor in faixas_turnos:
        fig.add_vrect(
            x0=inicio,
            x1=fim,
            fillcolor=cor,
            opacity=0.35,
            line_width=0,
            layer="below",
        )

        fig.add_annotation(
            x=hora_rotulo,
            y=0.98,
            xref="x",
            yref="paper",
            text=turno,
            showarrow=False,
            font={
                "size": 12,
                "color": "#34495E",
            },
            bgcolor="rgba(255, 255, 255, 0.70)",
            borderpad=3,
        )

    # Linhas pontilhadas às 14h e às 22h
    for limite in (7, 15):
        fig.add_shape(
            type="line",
            x0=limite,
            x1=limite,
            y0=0,
            y1=1,
            xref="x",
            yref="paper",
            line={
                "color": "rgba(52, 73, 94, 0.75)",
                "width": 1.5,
                "dash": "dot",
            },
            layer="above",
        )

    # Formata os valores do eixo Y como percentuais
    fig.update_yaxes(
        tickformat=".0%"
    )

    fig.update_layout(
        title_x=0.5,
        height=600,
        width=1000,
        showlegend=False,
        margin={
            "l": 80,
            "r": 60,
            "t": 90,
            "b": 70,
        },
    )

    fig.show()

In [ ]:
print('Casos onde "Qtd Rejeicao" > "Qtd Empacotado": ')
df_analise.query('`Qtd Rejeicao` > `Qtd Empacotado`')

## Variação de velocidade real por OP hora a hora

In [ ]:
df_analise = df.copy()
df_analise.sort_values(by='Data_Hora_Ordem_Producao', inplace=True)

fig = px.line(
    df_analise,
    x='Hora_Producao',
    y='Veloc Real',
    title='Variação de "Veloc Real" por OP hora a hora',
    color='OP Vertech',
    markers=True,
    category_orders={"Hora_Producao": ORDEM_HORAS_PRODUCAO},
    labels={
        'Hora_Producao': 'Hora do dia produtivo'
    }
)

fig.show()

## Variação de "Qtd Teorica Real" por OP hora a hora

In [ ]:
df_analise = df.copy()
df_analise.sort_values(by='Data_Hora_Ordem_Producao', inplace=True)

fig = px.line(
    df_analise,
    x='Hora_Producao',
    y='Qtd Teorica Real',
    title='Variação de "Qtd Teorica Real" por OP hora a hora',
    color='OP Vertech',
    markers=True,
    category_orders={"Hora_Producao": ORDEM_HORAS_PRODUCAO},
    labels={
        'Hora_Producao': 'Hora do dia produtivo'
    },
    hover_data=[
        'Empacotado %',
        'Qtd Empacotado',
        'Rejeição %',
        'Qtd Rejeicao',
        'Objetivo %',
        'Qtd Objetivo'
    ]
)

fig.show()

In [ ]:
df_analise = df.copy()
df_analise.sort_values(by='Data_Hora_Ordem_Producao', inplace=True)

# calculando: Veloc Real = Qtd Teorica Real / 60
df_analise['Calculo Veloc Real'] = df_analise['Qtd Teorica Real'] / 60

# comparando: "Caluclo Veloc Real" e "Veloc Real"
total_iguais = (df_analise['Veloc Real'] == df_analise['Calculo Veloc Real']).sum()
total_diferentes = (df_analise['Veloc Real'] != df_analise['Calculo Veloc Real']).sum()

print('Quantidade de valores que "Veloc Real" == "Calculo Veloc Real":')
display(total_iguais)

print()

print('Quantidade de valores que "Veloc Real" != "Calculo Veloc Real":')
display(total_diferentes)


## Variação do corte de gota por OP hora a hora

In [ ]:
df_analise = df.copy()
df_analise.sort_values(by='Data_Hora_Ordem_Producao', inplace=True)

fig = px.line(
    df_analise,
    x='Hora_Producao',
    y='Corte de Gota %',
    title='Variação de "Corte de Gota %" por OP hora a hora',
    color='OP Vertech',
    markers=True,
    category_orders={"Hora_Producao": ORDEM_HORAS_PRODUCAO},
    labels={
        'Hora_Producao': 'Hora do dia produtivo'
    }
)

fig.show()

## Gap entre "Emp - Rej %" e "Objetivo %" por OP hora a hora

In [ ]:
'''df_analise = df.copy()

df_analise.sort_values(
    by="Data_Hora_Ordem_Producao",
    inplace=True
)

fig = px.line(
    df_analise,
    x="Hora_Producao",
    y="Gap_Obj %",
    title='Gap entre "Emp - Rej %" e "Objetivo %" por OP hora a hora',
    color="OP Vertech",
    markers=True,
    category_orders={
        "Hora_Producao": ORDEM_HORAS_PRODUCAO
    },
    labels={
        "Hora_Producao": "Hora do dia produtivo",
        "Gap_Obj %": "Gap em relação ao objetivo",
        "OP Vertech": "OP Vertech",
    },
    hover_data=[
        "Qtd Gap_Obj",
        "Objetivo %",
        "Qtd Objetivo",
        "Empacotado %",
        "Qtd Empacotado",
        "Rejeição %",
        "Qtd Rejeicao",
        "Emp - Rej %",
        "Qtd Emp - Qtd Rej",
    ],
)

# Adiciona a linha de referência do objetivo
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
    line_width=2,
    annotation_text="Ponto de equilíbrio",
    annotation_position="top left",
)

fig.show()'''

df_analise = df.copy()

df_analise.sort_values(
    by="Data_Hora_Ordem_Producao",
    inplace=True
)

fig = px.line(
    df_analise,
    x="Hora_Producao",
    y="Gap_Obj %",
    title='Gap entre "Emp - Rej %" e "Objetivo %" por OP hora a hora',
    color="OP Vertech",
    markers=True,
    category_orders={
        "Hora_Producao": ORDEM_HORAS_PRODUCAO
    },
    labels={
        "Hora_Producao": "Hora do dia produtivo",
        "Gap_Obj %": "Gap em relação ao objetivo",
        "OP Vertech": "OP Vertech",
    },
    hover_data=[
        "Qtd Gap_Obj",
        "Objetivo %",
        "Qtd Objetivo",
        "Empacotado %",
        "Qtd Empacotado",
        "Rejeição %",
        "Qtd Rejeicao",
        "Emp - Rej %",
        "Qtd Emp - Qtd Rej",
    ],
)

# Faixas visuais dos turnos, seguindo a ordem do dia produtivo
faixas_turnos = [
    {
        "turno": "Turno A",
        "inicio": -0.5,
        "fim": 7,
        "hora_rotulo": "10:00",
        "cor": "#D6EAF8",
    },
    {
        "turno": "Turno B",
        "inicio": 7,
        "fim": 15,
        "hora_rotulo": "18:00",
        "cor": "#AED6F1",
    },
    {
        "turno": "Turno C",
        "inicio": 15,
        "fim": 23.5,
        "hora_rotulo": "02:00",
        "cor": "#85C1E9",
    },
]

# Adiciona as faixas coloridas e os nomes dos turnos
for faixa in faixas_turnos:
    fig.add_vrect(
        x0=faixa["inicio"],
        x1=faixa["fim"],
        fillcolor=faixa["cor"],
        opacity=0.35,
        line_width=0,
        layer="below",
    )

    fig.add_annotation(
        x=faixa["hora_rotulo"],
        y=0.98,
        xref="x",
        yref="paper",
        text=faixa["turno"],
        showarrow=False,
        font={
            "size": 12,
            "color": "#34495E",
        },
        bgcolor="rgba(255, 255, 255, 0.70)",
        borderpad=3,
    )

# Linhas pontilhadas nas mudanças de turno:
# Turno B às 14h e Turno C às 22h
limites_turnos = [7, 15]

for limite in limites_turnos:
    fig.add_shape(
        type="line",
        x0=limite,
        x1=limite,
        y0=0,
        y1=1,
        xref="x",
        yref="paper",
        line={
            "color": "rgba(52, 73, 94, 0.75)",
            "width": 1.5,
            "dash": "dot",
        },
        layer="above",
    )

# Linha de referência em que o desempenho
# líquido ficou exatamente igual ao objetivo
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
    line_width=2,
    annotation_text="Ponto de equilíbrio",
    annotation_position="top left",
)

fig.show()


In [ ]:
# Ordena uma única vez, sem modificar o df original
df_analise = df.sort_values(
    "Data_Hora_Ordem_Producao"
)

# Turno, início, fim, posição do rótulo e cor
faixas_turnos = [
    ("Turno A", -0.5, 7, "10:00", "#D6EAF8"),
    ("Turno B", 7, 15, "18:00", "#AED6F1"),
    ("Turno C", 15, 23.5, "02:00", "#85C1E9"),
]

# Cria um gráfico separado para cada OP
for op_vertech, dados_op in df_analise.groupby("OP Vertech"):
    fig = px.line(
        dados_op,
        x="Hora_Producao",
        y="Gap_Obj %",
        markers=True,
        color_discrete_sequence=["#3366CC"],
        category_orders={
            "Hora_Producao": ORDEM_HORAS_PRODUCAO
        },
        title=(
            f'Gap entre "Emp - Rej %" e "Objetivo %" '
            f"da OP {op_vertech} hora a hora"
        ),
        labels={
            "Hora_Producao": "Hora do dia produtivo",
            "Gap_Obj %": "Gap em relação ao objetivo",
        },
        hover_data=[
            "OP Vertech",
            "Qtd Gap_Obj",
            "Objetivo %",
            "Qtd Objetivo",
            "Empacotado %",
            "Qtd Empacotado",
            "Rejeição %",
            "Qtd Rejeicao",
            "Emp - Rej %",
            "Qtd Emp - Qtd Rej",
        ],
    )

    # Adiciona as faixas e os rótulos dos turnos
    for turno, inicio, fim, hora_rotulo, cor in faixas_turnos:
        fig.add_vrect(
            x0=inicio,
            x1=fim,
            fillcolor=cor,
            opacity=0.35,
            line_width=0,
            layer="below",
        )

        fig.add_annotation(
            x=hora_rotulo,
            y=0.98,
            xref="x",
            yref="paper",
            text=turno,
            showarrow=False,
            font={
                "size": 12,
                "color": "#34495E",
            },
            bgcolor="rgba(255, 255, 255, 0.70)",
            borderpad=3,
        )

    # Linhas pontilhadas às 14h e às 22h
    for limite in (7, 15):
        fig.add_shape(
            type="line",
            x0=limite,
            x1=limite,
            y0=0,
            y1=1,
            xref="x",
            yref="paper",
            line={
                "color": "rgba(52, 73, 94, 0.75)",
                "width": 1.5,
                "dash": "dot",
            },
            layer="above",
        )

    # Linha horizontal que representa o cumprimento exato do objetivo
    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="black",
        line_width=2,
        annotation_text="Ponto de equilíbrio",
        annotation_position="top left",
    )

    # Formata o eixo Y e mantém a escala automática por OP
    fig.update_yaxes(
        tickformat=".0%"
    )

    fig.update_layout(
        title_x=0.5,
        height=600,
        width=1000,
        showlegend=False,
        margin={
            "l": 80,
            "r": 60,
            "t": 90,
            "b": 70,
        },
    )

    fig.show()

In [ ]:
print('Registros onde "Objetivo %" == 0: ')
df.query('`Objetivo %` == 0')

## Percentual de horas abaixo do objetivo por OP

**Pergunta a responder:** Em qual percentual das horas cada máquina e prefixo ficaram abaixo do objetivo?

**Análise util para medir recorrência, por exemplo:**
- **Máquina A:** abaixo em 2 de 24 horas  → problema pontual
- **Máquina B:** abaixo em 15 de 24 horas → problema recorrente

In [ ]:
# A OP Vertech será a chave de cada produção
chave_analise = "OP Vertech"

# Identifica cada registro horário abaixo do objetivo
df["Abaixo do Objetivo"] = (
    df["Qtd Empacotado"]
    < df["Qtd Objetivo"]
)

# Quantidade total de registros horários da OP
df["Quantidade Total de Horas Trabalhadas"] = (
    df
    .groupby(
        chave_analise,
        dropna=False,
    )["Abaixo do Objetivo"]
    .transform("size")
)

# Quantidade de horas da OP abaixo do objetivo
df["Quantidade de Horas Abaixo do Objetivo"] = (
    df
    .groupby(
        chave_analise,
        dropna=False,
    )["Abaixo do Objetivo"]
    .transform("sum")
)

# Percentual das horas da OP abaixo do objetivo
df["Percentual de Horas Abaixo do Objetivo"] = (
    df
    .groupby(
        chave_analise,
        dropna=False,
    )["Abaixo do Objetivo"]
    .transform("mean")
    .mul(100)
)

# Cria uma tabela com uma linha por OP
resumo_horas_abaixo_objetivo_por_op = (
    df[
        [
            "OP Vertech",
            "Maquina",
            "Prefixo",
            "Quantidade Total de Horas Trabalhadas",
            "Quantidade de Horas Abaixo do Objetivo",
            "Percentual de Horas Abaixo do Objetivo",
        ]
    ]
    .drop_duplicates(subset=["OP Vertech"])
    .sort_values("OP Vertech")
    .reset_index(drop=True)
)

display(resumo_horas_abaixo_objetivo_por_op)
display(df)

In [ ]:
pct = "Percentual de Horas Abaixo do Objetivo"
total = "Quantidade Total de Horas Trabalhadas"
abaixo = "Quantidade de Horas Abaixo do Objetivo"
grupo = "OP"

df_grafico = (
    resumo_horas_abaixo_objetivo_por_op
    .assign(**{
        grupo: lambda dados: (
            dados["OP Vertech"].astype(str)
        )
    })
    .sort_values(pct)
)

fig = px.bar(
    df_grafico,
    x=pct,
    y=grupo,
    orientation="h",
    color=pct,
    text=pct,
    color_continuous_scale="RdYlGn_r",
    range_color=[0, 100],
    title="Percentual de Horas Abaixo do Objetivo por OP",
    labels={
        pct: "Horas abaixo do objetivo (%)",
        grupo: "OP Vertech",
        total: "Total de horas da OP",
        abaixo: "Horas abaixo do objetivo",
        "Maquina": "Máquina",
        "Prefixo": "Prefixo",
    },
    hover_data={
        "Maquina": True,
        "Prefixo": True,
        total: True,
        abaixo: True,
        pct: ":.2f",
    },
    template="plotly_white",
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False,
)

fig.update_xaxes(
    range=[0, 105],
    ticksuffix="%",
    dtick=10,
)

fig.update_yaxes(
    categoryorder="total ascending"
)

fig.update_layout(
    title_x=0.5,
    coloraxis_colorbar={
        "title": "Percentual",
        "ticksuffix": "%",
    },
    height=700,
    margin={
        "l": 100,
        "r": 60,
        "t": 80,
        "b": 60,
    },
)

fig.show()

## Duração da máquina abaixo do objetivo

**Pergunta a responder:** Houve horas consecutivas que a máquina ficou abaixo do objetivo? Se sim, quantas horas?

A ideia aqui é capturar comportamento da produção, assim podemos identificar casos como:
- Máquina parada para reformas
- Tempo de manutenções -> manutenções que demoraram para corrigir perdas (ponto debatido em reuniões)

In [ ]:
# Produção líquida adotada pela fábrica: empacotado menos rejeitado
coluna_producao = "Qtd Emp - Qtd Rej"

coluna_objetivo = "Qtd Objetivo"
chave_producao = "OP Vertech"

base_duracao = df[
    [
        chave_producao,
        "Maquina",
        "Prefixo",
        "Data Wht (dia)",
        "Data_metrica",
        "Data_Hora_Ordem_Producao",
        "Hora Hora Wht (6to6)",
        "Hora_Producao",
        "Ordem_Hora_Producao",
        coluna_producao,
        coluna_objetivo,
    ]
].copy()

base_duracao["Data_metrica"] = pd.to_datetime(
    base_duracao["Data_metrica"],
    errors="coerce",
)

# Mantém somente registros válidos para a análise
base_duracao = base_duracao.dropna(
    subset=[
        chave_producao,
        "Data_Hora_Ordem_Producao",
        coluna_producao,
        coluna_objetivo,
    ]
)

# Objetivo igual a zero não representa uma comparação válida
base_duracao = base_duracao[
    base_duracao[coluna_objetivo] > 0
].copy()

# Ordena cada OP seguindo o ciclo produtivo 07h até 06h
base_duracao = base_duracao.sort_values(
    [chave_producao, "Data_Hora_Ordem_Producao"]
).reset_index(drop=True)

In [ ]:
# Identifica as horas abaixo do objetivo
base_duracao["Abaixo do Objetivo"] = (
    base_duracao[coluna_producao]
    < base_duracao[coluna_objetivo]
)

# Calcula a perda de cada hora em unidades
base_duracao["Perda em Quantidade"] = (
    base_duracao[coluna_objetivo]
    - base_duracao[coluna_producao]
).clip(lower=0)

# Situação da hora anterior dentro da mesma OP
base_duracao["Abaixo na Hora Anterior"] = (
    base_duracao
    .groupby(chave_producao)["Abaixo do Objetivo"]
    .shift(fill_value=False)
)

# Intervalo operacional entre o registro atual e o anterior
base_duracao["Intervalo entre Horas"] = (
    base_duracao
    .groupby(chave_producao)["Data_Hora_Ordem_Producao"]
    .diff()
)

# Uma sequência começa quando a hora está abaixo e:
# - a hora anterior não estava abaixo; ou
# - não existe continuidade exata de uma hora
base_duracao["Inicio de Sequencia"] = (
    base_duracao["Abaixo do Objetivo"]
    & (
        ~base_duracao["Abaixo na Hora Anterior"]
        | base_duracao["Intervalo entre Horas"].ne(
            pd.Timedelta(hours=1)
        )
    )
)

# Cria um identificador para cada episódio dentro da OP
base_duracao["ID Episodio"] = (
    base_duracao
    .groupby(chave_producao)["Inicio de Sequencia"]
    .cumsum()
)

In [ ]:
episodios_abaixo_objetivo = (
    base_duracao[
        base_duracao["Abaixo do Objetivo"]
    ]
    .groupby(
        [chave_producao, "ID Episodio"],
        as_index=False,
    )
    .agg(
        Maquina=("Maquina", "first"),
        Prefixo=("Prefixo", "first"),
        Inicio_da_Perda=("Data_metrica", "first"),
        Inicio_Ordem_Producao=("Data_Hora_Ordem_Producao", "first"),
        Ultima_Hora_Abaixo=("Data_metrica", "last"),
        Ultima_Data_Producao=("Data Wht (dia)", "last"),
        Ultima_Ordem_Hora=("Ordem_Hora_Producao", "last"),
        Duracao_em_Horas=("Data_metrica", "size"),
        Perda_Total_em_Quantidade=(
            "Perda em Quantidade",
            "sum",
        ),
        Maior_Perda_Horaria=(
            "Perda em Quantidade",
            "max",
        ),
        Horas_do_Episodio=(
            "Data_metrica",
            lambda datas: (
                datas
                .dt.strftime("%d/%m/%Y %H:%M")
                .tolist()
            ),
        ),
    )
)

# O fim usa a próxima hora do ciclo 07h até 06h, mantendo o dia produtivo.
proxima_ordem = episodios_abaixo_objetivo["Ultima_Ordem_Hora"] + 1
proxima_hora = (proxima_ordem + 7) % 24
proxima_data_producao = (
    episodios_abaixo_objetivo["Ultima_Data_Producao"]
    + pd.to_timedelta(proxima_ordem // 24, unit="D")
)

episodios_abaixo_objetivo["Fim_da_Perda"] = (
    proxima_data_producao
    + pd.to_timedelta(proxima_hora, unit="h")
)

episodios_abaixo_objetivo = (
    episodios_abaixo_objetivo[
        [
            "OP Vertech",
            "Maquina",
            "Prefixo",
            "ID Episodio",
            "Inicio_Ordem_Producao",
            "Inicio_da_Perda",
            "Fim_da_Perda",
            "Duracao_em_Horas",
            "Perda_Total_em_Quantidade",
            "Maior_Perda_Horaria",
            "Horas_do_Episodio",
        ]
    ]
    .sort_values(
        ["Duracao_em_Horas", "Inicio_Ordem_Producao"],
        ascending=[False, True],
    )
    .drop(columns="Inicio_Ordem_Producao")
    .reset_index(drop=True)
)

display(episodios_abaixo_objetivo)

## Top OPs com mais horas abaixo do objetivo

**Pergunta a responder:** Quais foram as OPs que ficaram mais vezes abaixo do objetivo?

- Analise interessante para visualizar possiveis producoes que nao deveriam ter tantas quedas de desempenho

In [ ]:
base_contagem = df[
    [
        "OP Vertech",
        "Maquina",
        "Prefixo",
        "Data_metrica",
        "Qtd Emp - Qtd Rej",
        "Qtd Objetivo",
    ]
].copy()

# Mantém somente registros válidos
base_contagem = base_contagem.dropna(
    subset=[
        "OP Vertech",
        "Data_metrica",
        "Qtd Emp - Qtd Rej",
        "Qtd Objetivo",
    ]
)

# Desconsidera registros sem objetivo
base_contagem = base_contagem[
    base_contagem["Qtd Objetivo"] > 0
].copy()

# Identifica cada hora abaixo do objetivo
base_contagem["Abaixo do Objetivo"] = (
    base_contagem["Qtd Emp - Qtd Rej"]
    < base_contagem["Qtd Objetivo"]
)

# Cria o resumo por OP
contagem_abaixo_objetivo_por_op = (
    base_contagem
    .groupby(
        "OP Vertech",
        as_index=False,
    )
    .agg(
        Maquina=("Maquina", "first"),
        Prefixo=("Prefixo", "first"),
        Quantidade_Total_de_Horas=(
            "Data_metrica",
            "nunique",
        ),
        Quantidade_de_Vezes_Abaixo_do_Objetivo=(
            "Abaixo do Objetivo",
            "sum",
        ),
        Percentual_de_Horas_Abaixo_do_Objetivo=(
            "Abaixo do Objetivo",
            lambda valores: valores.mean() * 100,
        ),
    )
    .sort_values(
        "Quantidade_de_Vezes_Abaixo_do_Objetivo",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(contagem_abaixo_objetivo_por_op)

In [ ]:
quantidade_ops_exibidas = 10 # coloquei um valor alto para considerar todas as OPs
coluna_contagem = "Quantidade_de_Vezes_Abaixo_do_Objetivo"

df_grafico = (
    contagem_abaixo_objetivo_por_op
    .head(quantidade_ops_exibidas)
    .assign(
        OP=lambda dados: dados["OP Vertech"].astype(str)
    )
)

fig = px.bar(
    df_grafico,
    x=coluna_contagem,
    y="OP",
    orientation="h",
    color=coluna_contagem,
    color_continuous_scale="Reds",
    text=coluna_contagem,
    title=(
        f"Top {quantidade_ops_exibidas} OPs com Mais Horas "
        "Abaixo do Objetivo"
    ),
    labels={
        coluna_contagem: "Quantidade de horas abaixo",
        "OP": "OP Vertech",
        "Maquina": "Máquina",
        "Prefixo": "Prefixo",
        "Quantidade_Total_de_Horas": "Total de horas",
        "Percentual_de_Horas_Abaixo_do_Objetivo":
            "Percentual abaixo (%)",
    },
    hover_data={
        "Maquina": True,
        "Prefixo": True,
        "Quantidade_Total_de_Horas": True,
        "Percentual_de_Horas_Abaixo_do_Objetivo": ":.2f",
        coluna_contagem: True,
    },
    template="plotly_white",
)

fig.update_traces(
    texttemplate="%{text:.0f}",
    textposition="outside",
    cliponaxis=False,
)

fig.update_xaxes(
    dtick=1,
    title="Quantidade de horas abaixo do objetivo",
)

fig.update_yaxes(
    categoryorder="total ascending",
)

fig.update_layout(
    title_x=0.5,
    height=600,
    coloraxis_colorbar_title="Horas abaixo",
    margin={
        "l": 100,
        "r": 60,
        "t": 80,
        "b": 60,
    },
)

fig.show()

## Indicadores de "Emp. %", "Rej. %" e "Emp. - Rej. %" por OP diariamente

In [ ]:
import ipywidgets as widgets
from IPython.display import display

indicadores = [
    "Empacotado %",
    "Objetivo %",
    "Rejeição %",
    "Emp - Rejeitado %",
]

cores_indicadores = {
    "Empacotado %": "#3366CC",
    "Objetivo %": "#2CA02C",
    "Rejeição %": "#FF7F0E",
    "Emp - Rejeitado %": "#F2C80F",
}

base_grafico = desempenho_empacotamento_diario.copy()

base_grafico["OP Vertech"] = (
    base_grafico["OP Vertech"].astype(str)
)

# Ordena as OPs da menor para a maior performance
ops_ordenadas = (
    base_grafico
    .sort_values(
        "Emp - Rejeitado %",
        ascending=True,
    )["OP Vertech"]
    .tolist()
)

# Dropdown para selecionar uma OP
dropdown_op = widgets.Dropdown(
    options=ops_ordenadas,
    value=ops_ordenadas[0],
    description="OP Vertech:",
    style={
        "description_width": "initial"
    },
    layout=widgets.Layout(
        width="350px"
    ),
)


def exibir_grafico(op_selecionada):
    # Filtra a OP selecionada
    base_filtrada = (
        base_grafico[
            base_grafico["OP Vertech"]
            == op_selecionada
        ]
        .copy()
    )

    # Transforma as métricas em linhas
    dados_plot = base_filtrada.melt(
        id_vars=[
            "OP Vertech",
            "Data Wht (dia)",
            "Maquina",
            "Prefixo",
        ],
        value_vars=indicadores,
        var_name="Indicador",
        value_name="Percentual",
    )

    fig = px.bar(
        dados_plot,
        x="OP Vertech",
        y="Percentual",
        color="Indicador",
        barmode="group",
        category_orders={
            "Indicador": indicadores
        },
        color_discrete_map=cores_indicadores,
        title=(
            f"Resumo Diário de Empacotamento, Objetivo, "
            f"Rejeição e Desempenho da OP {op_selecionada}"
        ),
        labels={
            "OP Vertech": "OP",
            "Percentual": "Percentual",
            "Indicador": "Métrica",
            "Data Wht (dia)": "Data de produção",
            "Maquina": "Máquina",
            "Prefixo": "Prefixo",
        },
        hover_data={
            "Data Wht (dia)": True,
            "Maquina": True,
            "Prefixo": True,
            "Percentual": ":.2%",
        },
        template="plotly_white",
    )

    fig.update_traces(
        texttemplate="%{y:.1%}",
        textposition="outside",
        cliponaxis=False,
    )

    fig.update_yaxes(
        tickformat=".0%",
        title="Percentual",
        showgrid=True,
    )

    fig.update_xaxes(
        title="OP Vertech",
    )

    fig.update_layout(
        title_x=0.5,
        legend_title="Indicador",
        bargap=0.25,
        bargroupgap=0.08,
        height=600,
        width=900,
        margin={
            "l": 80,
            "r": 60,
            "t": 80,
            "b": 80,
        },
    )

    fig.show()


saida_grafico = widgets.interactive_output(
    exibir_grafico,
    {
        "op_selecionada": dropdown_op
    },
)

display(
    widgets.VBox([
        dropdown_op,
        saida_grafico,
    ])
)

In [ ]:
indicadores = [
    "Empacotado %",
    "Objetivo %",
    "Rejeição %",
    "Emp - Rejeitado %",
]

cores_indicadores = {
    "Empacotado %": "#3366CC",
    "Objetivo %": "#2CA02C",
    "Rejeição %": "#FF7F0E",
    "Emp - Rejeitado %": "#F2C80F",
}

# Prepara e ordena as OPs da menor para a maior performance
base_grafico = (
    desempenho_empacotamento_diario
    .assign(**{
        "OP Vertech": lambda dados:
            dados["OP Vertech"].astype(str)
    })
    .sort_values(
        "Emp - Rejeitado %",
        ascending=True,
    )
)

# Cria um gráfico separado para cada OP
for op_vertech, dados_op in base_grafico.groupby(
    "OP Vertech",
    sort=False,
):
    # Transforma os quatro indicadores em linhas
    dados_plot = dados_op.melt(
        id_vars=[
            "OP Vertech",
            "Data Wht (dia)",
            "Maquina",
            "Prefixo",
        ],
        value_vars=indicadores,
        var_name="Indicador",
        value_name="Percentual",
    )

    fig = px.bar(
        dados_plot,
        x="OP Vertech",
        y="Percentual",
        color="Indicador",
        barmode="group",
        category_orders={
            "Indicador": indicadores
        },
        color_discrete_map=cores_indicadores,
        title=(
            f"Resumo Diário de Empacotamento, Objetivo, "
            f"Rejeição e Desempenho da OP {op_vertech}"
        ),
        labels={
            "OP Vertech": "OP",
            "Percentual": "Percentual",
            "Indicador": "Métrica",
            "Data Wht (dia)": "Data de produção",
            "Maquina": "Máquina",
            "Prefixo": "Prefixo",
        },
        hover_data={
            "Data Wht (dia)": True,
            "Maquina": True,
            "Prefixo": True,
            "Percentual": ":.2%",
        },
        template="plotly_white",
    )

    fig.update_traces(
        texttemplate="%{y:.1%}",
        textposition="outside",
        cliponaxis=False,
    )

    fig.update_yaxes(
        tickformat=".0%",
        title="Percentual",
        showgrid=True,
    )

    fig.update_xaxes(
        title="OP Vertech",
    )

    fig.update_layout(
        title_x=0.5,
        legend_title="Indicador",
        bargap=0.25,
        bargroupgap=0.08,
        height=600,
        width=900,
        margin={
            "l": 80,
            "r": 60,
            "t": 80,
            "b": 80,
        },
    )

    fig.show()